<a href="https://colab.research.google.com/github/abegithub2024/abegithub2024/blob/main/RF_complete_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SOIL LOSS ESTIMATION USING RANDOM FOREST by SEI-Africa

**Abebe Heganno** (Intern)

**Anderson Kehbila (PhD)** Supervisor



In [ ]:
#SOIL LOSS ESTIMATION USING RANDOM FOREST IN SOUTHERN EWASO NG'IRO BASIN, KENYA
================================================================================
#Complete implementation for soil erosion prediction using machine learning.
Includes data preprocessing, model training, hyperparameter tuning, and spatial mapping.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                            classification_report, confusion_matrix)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# ============================================================
# 1. DATA PREPARATION (Replace with your actual data loading)
# ============================================================

def generate_sample_data(n_samples=1000):
    """
    Generate synthetic soil erosion data based on RUSLE framework.
    Replace this with your actual data loading code.
    """
    np.random.seed(42)

    data = {
        # RUSLE Factors
        'R_factor': np.random.uniform(50, 800, n_samples),      # Rainfall erosivity
        'K_factor': np.random.uniform(0.02, 0.69, n_samples),   # Soil erodibility
        'LS_factor': np.random.uniform(0.1, 30, n_samples),     # Slope length/steepness
        'C_factor': np.random.uniform(0.001, 1.0, n_samples),   # Cover management
        'P_factor': np.random.uniform(0.2, 1.0, n_samples),     # Conservation practice

        # Topographic features
        'elevation': np.random.uniform(0, 3000, n_samples),
        'slope_degrees': np.random.uniform(0, 45, n_samples),
        'aspect': np.random.uniform(0, 360, n_samples),
        'curvature': np.random.uniform(-10, 10, n_samples),

        # Vegetation indices
        'ndvi': np.random.uniform(-0.2, 0.9, n_samples),
        'land_cover': np.random.choice(['forest', 'agriculture', 'grassland', 'bare', 'urban'], n_samples),

        # Hydrological
        'drainage_density': np.random.uniform(0, 5, n_samples),
        'distance_to_river': np.random.uniform(0, 2000, n_samples),
        'runoff_coeff': np.random.uniform(0.1, 0.9, n_samples),
    }

    df = pd.DataFrame(data)

    # Calculate RUSLE baseline
    df['soil_loss_rusle'] = (df['R_factor'] * df['K_factor'] * df['LS_factor'] *
                             df['C_factor'] * df['P_factor'])

    # Add realistic noise
    df['soil_loss_actual'] = df['soil_loss_rusle'] * np.random.lognormal(0, 0.3, n_samples)

    # Create erosion classes
    def classify_erosion(loss):
        if loss < 5: return 'Low'
        elif loss < 25: return 'Moderate'
        elif loss < 50: return 'High'
        else: return 'Very High'

    df['erosion_class'] = df['soil_loss_actual'].apply(classify_erosion)

    return df

# Load or generate data
print("Loading data...")
df = generate_sample_data(1000)
print(f"Dataset shape: {df.shape}")

# ============================================================
# 2. DATA PREPROCESSING
# ============================================================

def preprocess_data(df):
    """Prepare features and targets for modeling."""

    # Encode categorical variables
    le_landcover = LabelEncoder()
    df['land_cover_encoded'] = le_landcover.fit_transform(df['land_cover'])

    le_erosion = LabelEncoder()
    df['erosion_class_encoded'] = le_erosion.fit_transform(df['erosion_class'])

    # Feature columns
    feature_cols = ['R_factor', 'K_factor', 'LS_factor', 'C_factor', 'P_factor',
                    'elevation', 'slope_degrees', 'aspect', 'curvature',
                    'ndvi', 'land_cover_encoded', 'drainage_density',
                    'distance_to_river', 'runoff_coeff']

    X = df[feature_cols]
    y_reg = df['soil_loss_actual']  # Continuous
    y_clf = df['erosion_class_encoded']  # Categorical

    # Split data
    X_train, X_test, y_reg_train, y_reg_test = train_test_split(
        X, y_reg, test_size=0.3, random_state=42)

    X_train_clf, X_test_clf, y_clf_train, y_clf_test = train_test_split(
        X, y_clf, test_size=0.3, random_state=42)

    return (X_train, X_test, y_reg_train, y_reg_test,
            X_train_clf, X_test_clf, y_clf_train, y_clf_test,
            feature_cols, le_erosion)

(X_train, X_test, y_reg_train, y_reg_test,
 X_train_clf, X_test_clf, y_clf_train, y_clf_test,
 feature_cols, le_erosion) = preprocess_data(df)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

# ============================================================
# 3. RANDOM FOREST REGRESSION
# ============================================================

def train_regression_model(X_train, y_train):
    """Train Random Forest Regressor for continuous soil loss prediction."""

    rf_reg = RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
        n_jobs=-1,
        oob_score=True
    )

    rf_reg.fit(X_train, y_train)
    return rf_reg

print("\nTraining Random Forest Regressor...")
rf_reg = train_regression_model(X_train, y_reg_train)

# Predictions and evaluation
y_pred = rf_reg.predict(X_test)
mse = mean_squared_error(y_reg_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_reg_test, y_pred)

print(f"\nRegression Results:")
print(f"  R² Score: {r2:.4f}")
print(f"  RMSE: {rmse:.4f} tons/ha/year")
print(f"  MAE: {np.mean(np.abs(y_reg_test - y_pred)):.4f}")
print(f"  OOB Score: {rf_reg.oob_score_:.4f}")

# Cross-validation
cv_scores = cross_val_score(rf_reg, X_train, y_reg_train, cv=5, scoring='r2')
print(f"  CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ============================================================
# 4. FEATURE IMPORTANCE ANALYSIS
# ============================================================

def plot_feature_importance(model, feature_names, save_path='feature_importance.png'):
    """Visualize feature importance."""

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\nTop 10 Important Features:")
    print(importance_df.head(10).to_string(index=False))

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=importance_df.head(10), y='feature', x='importance',
                palette='viridis', ax=ax)
    ax.set_title('Feature Importance for Soil Loss Prediction', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    return importance_df

importance_df = plot_feature_importance(rf_reg, feature_cols)

# ============================================================
# 5. HYPERPARAMETER TUNING
# ============================================================

def tune_hyperparameters(X_train, y_train):
    """Optimize Random Forest hyperparameters using RandomizedSearchCV."""

    param_dist = {
        'n_estimators': randint(50, 300),
        'max_depth': [None, 10, 20, 30, 40, 50],
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'max_features': ['sqrt', 'log2', None],
        'bootstrap': [True, False]
    }

    random_search = RandomizedSearchCV(
        RandomForestRegressor(random_state=42, n_jobs=-1),
        param_distributions=param_dist,
        n_iter=20,
        cv=5,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train, y_train)
    return random_search

print("\nTuning hyperparameters...")
random_search = tune_hyperparameters(X_train, y_reg_train)

print(f"\nBest Parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate tuned model
best_rf = random_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
r2_tuned = r2_score(y_reg_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_reg_test, y_pred_tuned))

print(f"\nTuned Model Performance:")
print(f"  R²: {r2_tuned:.4f} (Δ{r2_tuned - r2:+.4f})")
print(f"  RMSE: {rmse_tuned:.4f} (Δ{rmse - rmse_tuned:+.4f})")

# ============================================================
# 6. CLASSIFICATION APPROACH (Erosion Severity Classes)
# ============================================================

def train_classification_model(X_train, y_train):
    """Train Random Forest Classifier for erosion severity classes."""

    rf_clf = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )

    rf_clf.fit(X_train, y_train)
    return rf_clf

print("\nTraining Random Forest Classifier...")
rf_clf = train_classification_model(X_train_clf, y_clf_train)

# Classification evaluation
y_clf_pred = rf_clf.predict(X_test_clf)
accuracy = accuracy_score(y_clf_test, y_clf_pred)

print(f"\nClassification Results:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_clf_test, y_clf_pred,
                          target_names=le_erosion.classes_))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_clf_test, y_clf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_erosion.classes_,
            yticklabels=le_erosion.classes_)
plt.title('Confusion Matrix - Erosion Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

# ============================================================
# 7. VISUALIZATION OF RESULTS
# ============================================================

def plot_results(y_true, y_pred, save_path='prediction_results.png'):
    """Create comprehensive visualization of model performance."""

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Predicted vs Actual
    ax1 = axes[0, 0]
    ax1.scatter(y_true, y_pred, alpha=0.6, edgecolors='k', linewidth=0.5)
    ax1.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()],
             'r--', lw=2, label='Perfect Prediction')
    ax1.set_xlabel('Actual Soil Loss (tons/ha/year)')
    ax1.set_ylabel('Predicted Soil Loss (tons/ha/year)')
    ax1.set_title(f'Predicted vs Actual (R² = {r2:.3f})')
    ax1.legend()

    # 2. Residuals
    ax2 = axes[0, 1]
n    residuals = y_true - y_pred
    ax2.scatter(y_pred, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
    ax2.axhline(y=0, color='r', linestyle='--')
    ax2.set_xlabel('Predicted Soil Loss')
    ax2.set_ylabel('Residuals')
    ax2.set_title('Residual Plot')

    # 3. Distribution comparison
    ax3 = axes[1, 0]
    ax3.hist(y_true, bins=30, alpha=0.7, label='Actual', color='blue')
    ax3.hist(y_pred, bins=30, alpha=0.7, label='Predicted', color='red')
    ax3.set_xlabel('Soil Loss (tons/ha/year)')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Distribution Comparison')
    ax3.legend()

    # 4. Error distribution
    ax4 = axes[1, 1]
    ax4.hist(residuals, bins=30, color='green', alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Prediction Error')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Error Distribution')
    ax4.axvline(x=0, color='r', linestyle='--')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

plot_results(y_reg_test, y_pred)

# ============================================================
# 8. SPATIAL PREDICTION (For Raster Data)
# ============================================================

"""
For spatial mapping with raster data (requires rasterio), use this template:

import rasterio
from rasterio.transform import from_origin

def predict_spatial(raster_path, model, output_path):
    with rasterio.open(raster_path) as src:
        # Read all bands
        bands = src.read()
        profile = src.profile

        # Reshape for prediction (pixels x bands)
        n_bands, height, width = bands.shape
        pixels = bands.reshape(n_bands, -1).T

        # Predict
        predictions = model.predict(pixels)

        # Reshape back to raster
        output = predictions.reshape(height, width)

        # Save
        profile.update(dtype=rasterio.float32, count=1)
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(output.astype(rasterio.float32), 1)

    return output_path
"""

# ============================================================
# 9. SAVE MODEL
# ============================================================

import joblib

# Save models
joblib.dump(rf_reg, 'soil_loss_rf_regressor.pkl')
joblib.dump(rf_clf, 'soil_loss_rf_classifier.pkl')
joblib.dump(le_erosion, 'erosion_label_encoder.pkl')

print("\nModels saved:")
print("  - soil_loss_rf_regressor.pkl")
print("  - soil_loss_rf_classifier.pkl")
print("  - erosion_label_encoder.pkl")

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)

Key Components Explained:
1. **Data Structure**


RUSLE Factors: R (rainfall), K (soil), LS (topography), C (cover), P (practices)
Additional Features: NDVI, land cover, drainage, distance to water
Target: Continuous soil loss (tons/ha/year) or erosion classes
2.** Model Configuration**

In [ ]:
RandomForestRegressor(
    n_estimators=100,      # Number of trees
    max_depth=None,        # Expand until pure
    min_samples_split=5,   # Minimum samples to split
    max_features='sqrt',   # Features per split
    oob_score=True         # Out-of-bag validation
)

3. **Hyperparameter Tuning**


Uses RandomizedSearchCV for efficient optimization
Tests 20 random combinations with 5-fold CV
Optimizes: tree count, depth, split criteria, bootstrap
4. For Your Real Data
Replace the generate_sample_data() function with:

In [ ]:
# Load from CSV
df = pd.read_csv('your_soil_data.csv')

# Or load from shapefile (requires geopandas)
import geopandas as gpd
gdf = gpd.read_file('soil_samples.shp')
df = pd.DataFrame(gdf)

**5. Spatial Mapping**

